In [25]:
import pysubgroup as ps
from pysubgroup.datasets import get_titanic_data
import numpy as np
import pandas as pd


# Load example dataset
data = get_titanic_data()

# Define binary target: survived == True
target = ps.BinaryTarget('Survived', True)

# Define search space: all attributes except target
searchspace = ps.create_selectors(data, ignore=['Survived'])

In [26]:
data

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
151,152,1,1,"Pears, Mrs. Thomas (Edith Wearne)",female,22.0,1,0,113776,66.6000,C2,S
152,153,0,3,"Meo, Mr. Alfonzo",male,55.5,0,0,A.5. 11206,8.0500,NaN,S
153,154,0,3,"van Billiard, Mr. Austin Blyler",male,40.5,0,2,A/5. 851,14.5000,NaN,S
154,155,0,3,"Olsen, Mr. Ole Martin",male,NaN,0,0,Fa 265302,7.3125,NaN,S


In [41]:
import pandas as pd
import numpy as np
from pysubgroup.datasets import get_titanic_data

# Load and preprocess Titanic data
df = get_titanic_data()
categorical = ['Sex', 'Pclass', 'Embarked', 'SibSp', 'Parch']
df = df.dropna(subset=categorical + ['Survived', 'Age', 'Fare'])

# Bin Age and Fare for categorical subgrouping
df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'])
df['FareBin'] = pd.cut(df['Fare'], bins=[-1, 10, 30, 100, 600], labels=['Low', 'Mid', 'High', 'VeryHigh'])

features = ['Sex', 'Pclass', 'Embarked', 'SibSp', 'Parch', 'AgeBin', 'FareBin']
target_col = 'Survived'
target_value = True

class Subgroup:
    def __init__(self, conditions):
        self.conditions = conditions  # List of (feature, value) pairs

    def covers(self, df):
        mask = np.ones(len(df), dtype=bool)
        for feature, value in self.conditions:
            mask &= (df[feature] == value)
        return mask

    def extend(self, feature, value):
        return Subgroup(self.conditions + [(feature, value)])

    def __str__(self):
        return " AND ".join([f"{f}={v}" for f, v in self.conditions]) if self.conditions else "(all)"

def wracc(subgroup, df, target_col, target_value):
    mask = subgroup.covers(df)
    sg_size = mask.sum()
    if sg_size == 0:
        return 0.0
    p_sg = sg_size / len(df)
    p_t = np.mean(df[target_col] == target_value)
    p_t_sg = np.mean(df.loc[mask, target_col] == target_value)
    return p_sg * (p_t_sg - p_t)

def diversity(subgroup, subgroup_list, df):
    mask = subgroup.covers(df)
    overlaps = [mask & s.covers(df) for s in subgroup_list if s.conditions]
    if not overlaps:
        return 1
    overlap_ratios = [o.sum() / max(mask.sum(), 1) for o in overlaps]
    return 1 - np.mean(overlap_ratios)

def double_beam_sd(df, target_col, target_value, features, beam_width=5, max_depth=2):
    values = {f: df[f].unique() for f in features}
    quality_beam = [Subgroup([])]
    diversity_beam = [Subgroup([])]
    best_subgroups = []

    for depth in range(max_depth):
        candidates = []
        for subgroup in quality_beam + diversity_beam:
            used_conditions = set(subgroup.conditions)
            for f in features:
                for v in values[f]:
                    cond = (f, v)
                    if cond not in used_conditions:
                        candidates.append(subgroup.extend(f, v))
        # Remove duplicates
        candidates = list({str(s): s for s in candidates}.values())
        # Score candidates
        scored_quality = [(wracc(s, df, target_col, target_value), s) for s in candidates]
        scored_diversity = [(diversity(s, best_subgroups, df), s) for s in candidates]
        # Select top-k for each beam
        quality_beam = [s for _, s in sorted(scored_quality, key=lambda x: x[0], reverse=True)[:beam_width]]
        diversity_beam = [s for _, s in sorted(scored_diversity, key=lambda x: x[0], reverse=True)[:beam_width]]
        # Update best_subgroups
        best_subgroups.extend(quality_beam)
        best_subgroups = list({str(s): s for s in best_subgroups}.values())
    # Return best subgroups found by quality
    return sorted(best_subgroups, key=lambda s: wracc(s, df, target_col, target_value), reverse=True)[:beam_width]

# Run Double Beam SD
subgroups = double_beam_sd(df, target_col, target_value, features, beam_width=5, max_depth=2)
for idx, sg in enumerate(subgroups):
    mask = sg.covers(df)
    print(f"Subgroup {idx+1}: {sg}")
    print(f"  WRAcc: {wracc(sg, df, target_col, target_value):.3f} | Size: {mask.sum()} | SurvivalRate: {df.loc[mask, target_col].mean():.2f}")

Subgroup 1: Sex=female
  WRAcc: 0.130 | Size: 46 | SurvivalRate: 0.67
Subgroup 2: Sex=female AND Embarked=S
  WRAcc: 0.097 | Size: 37 | SurvivalRate: 0.65
Subgroup 3: Sex=female AND Parch=0
  WRAcc: 0.092 | Size: 33 | SurvivalRate: 0.67
Subgroup 4: Sex=female AND AgeBin=YoungAdult
  WRAcc: 0.088 | Size: 22 | SurvivalRate: 0.82
Subgroup 5: AgeBin=YoungAdult AND Sex=female
  WRAcc: 0.088 | Size: 22 | SurvivalRate: 0.82


In [35]:
# Define WRAcc quality function
qf = ps.WRAccQF()

# Define subgroup discovery task
task = ps.SubgroupDiscoveryTask(
    data,
    target,
    searchspace,
    qf,
    result_set_size=10,
    depth=2
)

In [39]:
# ... previous code ...
result = ps.BeamSearch().execute(task)

# Print the discovered subgroups (DataFrame method)
df_result = result.to_dataframe()
for i, row in df_result.iterrows():
    print(f"Subgroup {i+1}: {row['subgroupDescription']}")
    print(f"  WRAcc: {row['quality']:.3f} | Size: {row['subgroup_size']} | Positives: {row['positive_subgroup_size']}")

KeyError: 'subgroupDescription'

In [40]:
# Run the subgroup discovery algorithm
result = ps.BeamSearch().execute(task)

# Convert results to DataFrame and inspect columns
df_result = result.to_dataframe()
print("Result columns:", df_result.columns)

# Choose correct description column
desc_col = None
for possible in ['subgroupDescription', 'description']:
    if possible in df_result.columns:
        desc_col = possible
        break
if desc_col is None:
    desc_col = df_result.columns[0]  # fallback to the first column

# Print the discovered subgroups
for i, row in df_result.iterrows():
    print(f"Subgroup {i+1}: {row[desc_col]}")
    print(f"  WRAcc: {row['quality']:.3f} | Size: {row['subgroup_size']} | Positives: {row['positive_subgroup_size']}")

Result columns: Index(['quality', 'subgroup', 'size_sg', 'size_dataset', 'positives_sg',
       'positives_dataset', 'size_complement', 'relative_size_sg',
       'relative_size_complement', 'coverage_sg', 'coverage_complement',
       'target_share_sg', 'target_share_complement', 'target_share_dataset',
       'lift'],
      dtype='object')
Subgroup 1: 0.13214990138067062


KeyError: 'subgroup_size'